In [1]:
import glob
import pandas as pd

# Load files

In [2]:
files = glob.glob("data/results_*.jsonl")
files

['data/results_refugee.jsonl',
 'data/results_unhcr.jsonl',
 'data/results_prwp.jsonl']

In [3]:
data = pd.concat([pd.read_json(x, lines=True) for x in files]).reset_index(drop=True)
data

,snapshot_id,source,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error
0,027_Jordan-Emergency-Food-Security-Project_fig...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 7168, 'input_tokens_details':...","{'input_tokens': 7168, 'output_tokens': 2431, ...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
1,054_Sudan-Basic-Education-Emergency-Support-Pr...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 13799, 'input_tokens_details'...","{'input_tokens': 13799, 'output_tokens': 1576,...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
2,134_PAD7340ARABIC00n0220020140Arabic01_figure_...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 13286, 'input_tokens_details'...","{'input_tokens': 13286, 'output_tokens': 1460,...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
3,004_BOSIB-87c444de-4797-4bf9-b654-4932a7fb0112...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 8299, 'input_tokens_details':...","{'input_tokens': 8299, 'output_tokens': 2116, ...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
4,202_multi0page_figure_000.png,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""geographic_scope...","{'input_tokens': 12161, 'input_tokens_details'...","{'input_tokens': 12161, 'output_tokens': 2134,...",completed,NaN,{'fields': [{'metadata_field': 'geographic_sco...,NaN
5,200_multi-page_table_004.png,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 9304, 'input_tokens_details':...","{'input_tokens': 9304, 'output_tokens': 1526, ...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
6,200_multi-page_table_003.png,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 10634, 'input_tokens_details'...","{'input_tokens': 10634, 'output_tokens': 1530,...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
7,054_Sudan-Basic-Education-Emergency-Support-Pr...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 13554, 'input_tokens_details'...","{'input_tokens': 13554, 'output_tokens': 1939,...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
8,195_multi-page_table_012.png,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 14962, 'input_tokens_details'...","{'input_tokens': 14962, 'output_tokens': 1525,...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
9,120_PAD1205-ARABIC-PAD-PP152646-PUBLIC-Box3932...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 5548, 'input_tokens_details':...","{'input_tokens': 5548, 'output_tokens': 1514, ...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN


In [4]:
data[data["error"].notna()]

,snapshot_id,source,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error


# Preprocessing

In [5]:
pd.set_option('display.max_colwidth', None)

In [6]:
df = data[["snapshot_id", "source", "model", "parsed_output"]].copy()
df["parsed_output"] = df["parsed_output"].apply(
    lambda x: x.get("fields", []) if isinstance(x, dict) else []
)

df = df.explode("parsed_output").reset_index(drop=True)
df = df.join(pd.json_normalize(df["parsed_output"])).drop(columns="parsed_output")

df

,snapshot_id,source,model,metadata_field,observed_value,description,source_level,discovery_value,reasoning
0,027_Jordan-Emergency-Food-Security-Project_figure_001.png,refugee,gpt-5.5,figure_title,Figure 2: Share of food expenditure in total expenditure p.c. (%),The title or caption identifying the figure and its subject.,snapshot,high,The figure title provides the most direct searchable description of the snapshot content.
1,027_Jordan-Emergency-Food-Security-Project_figure_001.png,refugee,gpt-5.5,indicator_name,Share of food expenditure in total expenditure p.c.,The main measure or indicator represented in the figure.,snapshot,high,Indicator-level metadata enables retrieval of figures by the specific economic or food-security measure shown.
2,027_Jordan-Emergency-Food-Security-Project_figure_001.png,refugee,gpt-5.5,subject_domain,Food expenditure; food security,The thematic domain or topical area represented by the snapshot.,both,high,The figure title and document metadata both connect the snapshot to food expenditure and food security topics.
3,027_Jordan-Emergency-Food-Security-Project_figure_001.png,refugee,gpt-5.5,geographic_scope,"Jordan, inferred from parent document title and project name; not visible in snapshot","The country, region, or geographic area to which the figure data applies.",document,high,Geographic context is essential for discovery because it is not shown in the figure itself.
4,027_Jordan-Emergency-Food-Security-Project_figure_001.png,refugee,gpt-5.5,time_period,Not identifiable from this snapshot,"The date, year, or period covered by the data in the figure.",snapshot,high,"A data time period would be highly useful for comparison, but it is not visible in the figure."
...,...,...,...,...,...,...,...,...,...
432,document_10087951_table_005.png,prwp,gpt-5.5,intervention_program,Nutrition Enhancement Program,The program or intervention evaluated by the table or document.,document,high,Program metadata links the table’s estimates to the intervention being evaluated.
433,document_10087951_table_005.png,prwp,gpt-5.5,geographic_scope,Senegal,"The country, region, or place to which the table’s analysis applies.",document,high,Geographic scope is essential for filtering and contextualizing empirical results.
434,document_10087951_table_005.png,prwp,gpt-5.5,population_group,"Children; inferred from child nutrition context and indicators such as colostrum, Vitamin A, diarrhea, and cough",The population or demographic group represented by the table’s indicators or analysis.,both,high,Population-group metadata improves discovery of child health and nutrition evidence.
435,document_10087951_table_005.png,prwp,gpt-5.5,research_design,Large-scale randomized community intervention with planned treatment status,The study or evaluation design underlying the reported estimates.,both,medium,Research-design metadata helps users find tables from randomized or quasi-experimental evaluations.


In [7]:
df.to_csv("examples.csv", index=False)